# 09 — Quantized Multi-Hop Reasoning on Tiny Aya (3.35B)
## قياس أثر التكميم والاستدلال متعدد الخطوات في النماذج متعددة اللغات أصلاً (Tiny Aya Earth)

> **The Empirical Puzzle & Research Hypothesis:**
> In notebooks `07` and `08`, we evaluated Google's `gemma-2-2b` on 2-hop factual reasoning across English, Modern Standard Arabic (MSA), and Egyptian Arabic (Masri). We uncovered two critical vulnerabilities:
> 1. **The L22 English Pivot:** Despite receiving Arabic prompts, Gemma-2 resolves multi-hop factual relations ($[\text{Suez Canal}] \to [\text{Egypt}] \to [\text{Cairo}]$) by routing through an internal English-dominated subspace at Layer 22 ($P(\text{' Cairo'}) = 0.975$) before attempting a late cross-lingual un-embedding projection.
> 2. **Asymmetric Quantization Penalty:** Low-bit weight quantization (Int4 RTN) shattered this late cross-lingual projection, causing a $-65.8\%$ factual drop in MSA and a $-72.9\%$ drop in Masri, with Masri collapsing into a conversational question-mark attractor ($P('?') = 0.994$).
>
> **The Core Question for Tiny Aya:**
> Tiny Aya (3.35B, `CohereLabs/tiny-aya-earth`) was pretrained natively on over 70 languages with massive African and Middle Eastern data representation. 
> - **H1 (Pivot Elimination):** Does native multilingual pre-training bypass the English lingua franca pivot, resolving Arabic facts directly within an Arabic semantic subspace?
> - **H2 (Dialectal Quantization Resilience):** Does native multilingual representation confer structural resilience against low-bit quantization (Int8 and Int4), preventing the dialectal collapse seen in Gemma-2-2B?
> - **H3 (RTN vs Distribution-Aware NF4):** Does information-theoretically optimal 4-bit quantization (NormalFloat4) protect fragile dialectal representations where naive Round-To-Nearest (RTN) fails?

### Step 1: Environment Configuration & ROCm Execution Sanity Check

We initialize the execution environment on AMD Strix Halo using ROCm. We configure PyTorch to use `HSA_OVERRIDE_GFX_VERSION=11.0.0` to target the RDNA 3.5 iGPU architecture cleanly, disable tokenizers parallelism to eliminate fork warnings, and assert CUDA/ROCm device availability.

In [1]:
from __future__ import annotations

import os
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

os.environ.setdefault("HSA_OVERRIDE_GFX_VERSION", "11.0.0")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__}")
print(f"Execution Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")


PyTorch version: 2.5.1+rocm6.2
Execution Device: cuda
Device Name: AMD Radeon Graphics
